# 02 – Finding pain points with sentence-level topic clustering
Pipeline:
1. Clean reviews (HTML entities, Steam formatting tags, links)
2. Keep only reviews actually written in English
3. Split reviews into sentences
4. Keep only negative sentences (sentiment model)
5. Cluster those sentences into pain-point topics (BERTopic)
6. Rank topics by how many reviews mention them and how much the community agreed

Whole reviews mix many topics, so clustering them produces one vague blob. Sentences usually carry one complaint each.

In [1]:
import html
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)
DATA = Path("../data")

conn = sqlite3.connect(DATA / "reviews.db")
df = pd.read_sql("SELECT * FROM reviews", conn)
df["hours_at_review"] = df["playtime_at_review"].fillna(0) / 60
neg = df[df["voted_up"] == 0].copy()
print(f"{len(neg)} negative reviews")

5096 negative reviews


## 1. Cleaning

In [2]:
BBCODE = re.compile(r"\[/?[a-z0-9*]+(?:=[^\]]*)?\]", re.I)   # [b], [h1], [url=...]
URL = re.compile(r"https?://\S+")

def clean(text: str) -> str:
    text = html.unescape(text or "").replace("\xa0", " ")
    text = BBCODE.sub(" ", text)
    text = URL.sub(" ", text)
    text = re.sub(r"[ \t]+", " ", text)   # keep newlines: they often separate points
    return text.strip()

neg["clean"] = neg["text"].map(clean)
neg = neg[neg["clean"].str.len() >= 50]
print(f"{len(neg)} substantial negative reviews after cleaning")

3896 substantial negative reviews after cleaning


## 2. Language filter
Steam's `language` field is what the reviewer *selected*, not what they wrote, so we detect it ourselves.

In [3]:
from langdetect import DetectorFactory, detect

DetectorFactory.seed = 0   # langdetect is random otherwise

def is_english(text: str) -> bool:
    try:
        return detect(text) == "en"
    except Exception:
        return False

neg = neg[neg["clean"].map(is_english)].reset_index(drop=True)
print(f"{len(neg)} English negative reviews")

3843 English negative reviews


## 3. Split into sentences
Very short fragments ("10/10", "no.") carry no complaint; very long ones are usually run-on rants.

In [4]:
SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")

def split_sentences(text: str, min_words: int = 5, max_words: int = 60) -> list[str]:
    out = []
    for s in SPLIT.split(text):
        s = s.strip(" -*•\t")
        if min_words <= len(s.split()) <= max_words:
            out.append(s)
    return out

sentences = (
    neg[["review_id", "votes_up", "hours_at_review", "clean"]]
    .assign(sentence=neg["clean"].map(split_sentences))
    .explode("sentence")
    .dropna(subset=["sentence"])
    .drop(columns="clean")
    .reset_index(drop=True)
)
print(f"{len(sentences)} sentences from {sentences['review_id'].nunique()} reviews")

27608 sentences from 3809 reviews


## 4. Keep negative sentences
Negative reviews still contain praise ("the story is great, but..."). A sentiment model tags each sentence; results are cached because this is the slowest step. Delete `neg_sentences.pkl` after collecting new data.

In [ ]:
from transformers import pipeline

cache = DATA / "neg_sentences.pkl"

if cache.exists():
    sentences = pd.read_pickle(cache)
else:
    sentiment = pipeline(
        "sentiment-analysis",
        model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True,
    )
    preds = sentiment(sentences["sentence"].tolist(), batch_size=64)
    sentences["sentiment"] = [p["label"] for p in preds]
    sentences["confidence"] = [p["score"] for p in preds]
    sentences.to_pickle(cache)

complaints = sentences[
    (sentences["sentiment"] == "NEGATIVE") & (sentences["confidence"] >= 0.9)
].reset_index(drop=True)
print(f"{len(complaints)} complaint sentences")
complaints.sample(10, random_state=0)[["sentence", "confidence"]]

c:\Users\user\Desktop\Visual Studio Projects\Game Review Analyzer\game-review-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\user\Desktop\Visual Studio Projects\Game Review Analyzer\game-review-analyzer\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#l

## 5. Cluster complaints

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
docs = complaints["sentence"].tolist()
embeddings = embedder.encode(docs, show_progress_bar=True)

Generic words ("game", "like") and the game's own name dominate topic labels without adding meaning, so they're removed from the *labels* only. Edit `GAME_WORDS` when analyzing a different game.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from umap import UMAP

GENERIC_WORDS = {"game", "games", "like", "just", "really", "play", "playing", "played",
                 "time", "good", "bad", "don", "doesn", "didn", "isn", "ve", "ll", "im",
                 "thing", "things", "lot", "way", "make", "feel", "feels", "want"}
GAME_WORDS = {"baldur", "baldurs", "gate", "bg3", "bg"}
stop_words = list(ENGLISH_STOP_WORDS | GENERIC_WORDS | GAME_WORDS)

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=42)
vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2), min_df=3)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    vectorizer_model=vectorizer,
    min_topic_size=40,   # main knob: higher = fewer, broader topics
    verbose=True,
)
topics, _ = topic_model.fit_transform(docs, embeddings)
complaints["topic"] = topics
topic_model.get_topic_info().head(25)

## 6. Inspect topics
Topic modeling always outputs *something*. Read real examples to judge whether each topic is a genuine pain point.

In [ ]:
info = topic_model.get_topic_info()
for topic_id in info["Topic"].head(16):
    if topic_id == -1:
        continue
    words = ", ".join(w for w, _ in topic_model.get_topic(topic_id)[:6])
    print(f"=== Topic {topic_id}: {words} ===")
    for s in complaints.loc[complaints["topic"] == topic_id, "sentence"].sample(3, random_state=0):
        print("-", s)
    print()

## 7. Rank pain points
Counted per **review** (not sentence) so one long rant can't inflate a topic.

In [ ]:
per_review = complaints[complaints["topic"] != -1].drop_duplicates(["topic", "review_id"])

ranking = per_review.groupby("topic").agg(
    reviews=("review_id", "size"),
    total_upvotes=("votes_up", "sum"),
    median_hours=("hours_at_review", "median"),
)
ranking["share_of_negative_reviews"] = (ranking["reviews"] / len(neg)).round(3)
ranking["label"] = [", ".join(w for w, _ in topic_model.get_topic(t)[:4]) for t in ranking.index]
ranking = ranking.sort_values("reviews", ascending=False)
ranking

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

## 8. Save

In [ ]:
complaints[["review_id", "sentence", "topic"]].to_csv(DATA / "complaint_topics.csv", index=False)
ranking.to_csv(DATA / "pain_point_ranking.csv")
print("Saved.")

## Notes
Which topics are real pain points? Which should be merged? Is anything obviously missing?